# ⚙️ Project 14.3 — Production Optimiser: DP & Greedy Decision Suite
**DSA for Mechatronics · Week 14 — Review & Final Project**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.

**Topics integrated:** Max Product Subarray DP (W12) · Coin Change DP (W12) · Activity Selection Greedy (W13) · Fractional Knapsack Greedy (W13) · DP vs Greedy comparison (W14)


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, heapq, math
from collections import defaultdict, deque, Counter, OrderedDict
from functools import lru_cache
import bisect

_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A production line runs four optimisation problems. For each, we must decide:
*should we use DP or greedy?*

1. **Max product subarray** — find the contiguous subarray with the largest product
   (DP: track both max AND min because negatives flip sign)
2. **Coin change — minimum coins** — fewest tokens to make a target amount
   (Greedy FAILS for general denominations; DP gives the correct answer)
3. **Activity selection** — maximum non-overlapping production runs
   (Greedy WORKS: sort by end time, always take the earliest-finishing run)
4. **Fractional batch loading** — load the most valuable fractional batches
   into a fixed-capacity container (Greedy WORKS for fractional items; DP needed for 0/1)
   We demonstrate the gap: run both and compare.


## Step 1 — Generate production datasets

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_PROD   = random.randint(10, 16)   # product subarray length
N_COINS  = random.randint(3, 5)     # denominations
COIN_TGT = random.randint(20, 40)
N_RUNS   = random.randint(10, 15)   # activity selection intervals
T_MAX    = random.randint(20, 30)
N_BATCH  = random.randint(7, 11)    # fractional knapsack items
CAPACITY = random.randint(30, 50)

# Max product: mix of positives, negatives, occasional zero
PROD_ARR = [random.choice([-1,1]) * random.randint(1, 5)
            for _ in range(N_PROD)]
if random.random() < 0.3:  # sometimes insert a zero for interest
    PROD_ARR[random.randint(0, N_PROD-1)] = 0

# Coin denominations: guarantee coins=[1,...] so always feasible
COINS = sorted(set([1] + [random.randint(2, 15)
                           for _ in range(N_COINS - 1)]))

# Production runs (intervals)
def rand_iv(t_max):
    s = random.randint(0, t_max-2)
    return (s, random.randint(s+1, t_max))
RUNS = [rand_iv(T_MAX) for _ in range(N_RUNS)]

# Batch items (name, weight, value)
BATCHES = [(f"B{i+1}", round(random.uniform(2, 12), 1),
            round(random.uniform(10, 60), 1)) for i in range(N_BATCH)]

print(f"Production datasets:")
print(f"  Product array (n={N_PROD}) : {PROD_ARR}")
print(f"  Coin denominations         : {COINS}  target={COIN_TGT}")
print(f"  Production runs (n={N_RUNS})  : {RUNS}")
print(f"  Batches (n={N_BATCH}, cap={CAPACITY}) :")
for name, w, v in BATCHES:
    print(f"    {name}: w={w:.1f}  v={v:.1f}  ratio={v/w:.3f}")


## Step 2 — Max product subarray (DP)

In [ ]:
def max_product_subarray(nums):
    """
    Maximum product of any contiguous subarray.

    WHY DP, not greedy?
    A negative number × negative number = positive — so a large
    negative product can become the new maximum if we encounter another
    negative. We must track BOTH the current maximum AND minimum product
    ending at each position.

    Recurrence (at position i, current element = x):
      max_here = max(x, max_here * x, min_here * x)
      min_here = min(x, max_here_prev * x, min_here_prev * x)

    The global answer is max(max_here) over all i.
    O(n) time, O(1) space.
    """
    if not nums: return 0, []
    max_prod = min_prod = global_max = nums[0]
    best_end = best_start = cur_start = 0
    trace = [(nums[0], nums[0], nums[0], 0)]

    for i in range(1, len(nums)):
        x = nums[i]
        # When x < 0, max and min swap roles
        candidates_max = (x, max_prod * x, min_prod * x)
        candidates_min = (x, max_prod * x, min_prod * x)
        prev_max = max_prod
        max_prod = max(candidates_max)
        min_prod = min(candidates_min)
        if max_prod == x:           # started fresh
            cur_start = i
        if max_prod > global_max:
            global_max  = max_prod
            best_end    = i
            best_start  = cur_start
        trace.append((x, max_prod, min_prod, global_max))

    subarray = nums[best_start:best_end+1]
    return global_max, subarray, best_start, best_end, trace

mp_val, mp_sub, mp_s, mp_e, mp_trace = max_product_subarray(PROD_ARR)

# Brute-force verify
bf_max = max(
    (lambda s: s if s != 0 else s)(
        __import__('math').prod(PROD_ARR[i:j+1]))
    for i in range(N_PROD) for j in range(i, N_PROD))

print(f"Max product subarray (DP) — n={N_PROD}:")
print(f"  Array   : {PROD_ARR}")
print()
print(f"  {'i':>3}  {'val':>5}  {'max_here':>10}  {'min_here':>10}  {'global_max':>12}")
print(f"  {'─'*3}  {'─'*5}  {'─'*10}  {'─'*10}  {'─'*12}")
for i, (x, mx, mn, gm) in enumerate(mp_trace[:min(10, N_PROD)]):
    print(f"  {i:>3}  {x:>5}  {mx:>10}  {mn:>10}  {gm:>12}")
if N_PROD > 10:
    print(f"  ... ({N_PROD-10} more rows)")
print()
prod_check = __import__('math').prod(mp_sub)
print(f"  Max product subarray: {mp_sub}  (indices [{mp_s},{mp_e}])")
print(f"  Product             : {prod_check}")
print(f"  DP result           : {mp_val}")
print(f"  Brute-force verify  : {bf_max}")
print(f"  Correct             : {'✅ YES' if mp_val == bf_max else '❌ NO'}")


## Step 3 — Coin change: why greedy fails, DP succeeds

In [ ]:
def coin_change_dp(coins, target):
    """
    Minimum coins to make target — O(target × n_coins) DP.
    This is CORRECT for any denominations.
    See W12 P14.4 for full explanation.
    """
    dp      = [float('inf')] * (target + 1)
    dp[0]   = 0
    parent  = [-1] * (target + 1)
    for i in range(1, target + 1):
        for c in coins:
            if c <= i and dp[i-c] + 1 < dp[i]:
                dp[i]    = dp[i-c] + 1
                parent[i]= c
    used = []
    amt  = target
    while amt > 0 and parent[amt] != -1:
        used.append(parent[amt]); amt -= parent[amt]
    return (dp[target] if dp[target] != float('inf') else -1), used

def coin_change_greedy(coins, target):
    """
    Greedy coin change: always take the largest coin ≤ remaining.
    FAILS for non-canonical coin systems (e.g. coins=[1,3,4], target=6
    → greedy gives 4+1+1=3 coins, DP gives 3+3=2 coins).
    """
    remaining = target
    used = []
    for c in sorted(coins, reverse=True):
        while remaining >= c:
            used.append(c); remaining -= c
    return (len(used) if remaining == 0 else -1), used

dp_count,  dp_coins  = coin_change_dp(COINS, COIN_TGT)
gr_count,  gr_coins  = coin_change_greedy(COINS, COIN_TGT)

print(f"Coin change (denominations={COINS}, target={COIN_TGT}):")
print()
print(f"  DP result    : {dp_count} coins  → {sorted(dp_coins, reverse=True)}")
print(f"  Greedy result: {gr_count} coins  → {sorted(gr_coins, reverse=True)}")
print(f"  DP ≤ Greedy  : {'✅ YES (DP is optimal)' if dp_count <= gr_count else '❌ NO (unexpected)'}")
print()
# Construct a counterexample where greedy fails
# Use denominations [1, 3, 4] with target 6
demo_coins = [1, 3, 4]; demo_tgt = 6
d_dp_n, d_dp_c  = coin_change_dp(demo_coins, demo_tgt)
d_gr_n, d_gr_c  = coin_change_greedy(demo_coins, demo_tgt)
print(f"  Counterexample where greedy fails:")
print(f"    coins={demo_coins}, target={demo_tgt}")
print(f"    Greedy: {d_gr_n} coins {sorted(d_gr_c, reverse=True)}  (takes 4, then 1+1)")
print(f"    DP    : {d_dp_n} coins {sorted(d_dp_c, reverse=True)}  (takes 3+3)")
print(f"    Greedy suboptimal: {'✅ demonstrated' if d_gr_n > d_dp_n else 'same result here'}")


## Step 4 — Activity selection (greedy) + fractional knapsack (greedy) vs 0/1 DP

In [ ]:
def activity_selection(intervals):
    """Greedy: sort by finish time, take non-overlapping. O(n log n)."""
    s = sorted(enumerate(intervals), key=lambda x: x[1][1])
    selected = []; last_end = -1
    for idx, (start, end) in s:
        if start >= last_end:
            selected.append((idx, start, end)); last_end = end
    return selected

def fractional_knapsack(items, cap):
    """Greedy: sort by v/w ratio. O(n log n)."""
    s = sorted(items, key=lambda x: x[2]/x[1], reverse=True)
    total = 0.0; taken = []; rem = cap
    for name, w, v in s:
        if rem <= 0: break
        frac = min(1.0, rem/w)
        taken.append((name, frac, frac*v))
        total += frac*v; rem -= frac*w
    return total, taken

def dp_knapsack_01(items, cap):
    """0/1 DP knapsack with int weights (×10 for 1 decimal)."""
    cap_i   = int(cap*10)
    items_i = [(n, int(w*10), v) for n, w, v in items]
    dp = [0.0]*(cap_i+1)
    for _, wi, vi in items_i:
        for c in range(cap_i, wi-1, -1):
            dp[c] = max(dp[c], dp[c-wi]+vi)
    return dp[cap_i]

act_sel   = activity_selection(RUNS)
fk_val, fk_taken   = fractional_knapsack(BATCHES, CAPACITY)
dp01_val  = dp_knapsack_01(BATCHES, CAPACITY)

print(f"Activity selection (n={N_RUNS} runs, T_MAX={T_MAX}):")
sel_ok = all(act_sel[i+1][1] >= act_sel[i][2] for i in range(len(act_sel)-1))
print(f"  Selected {len(act_sel)} of {N_RUNS} runs: {[(i,s,e) for i,s,e in act_sel]}")
print(f"  Non-overlapping: {'✅ YES' if sel_ok else '❌ NO'}")
print()

print(f"Fractional knapsack vs 0/1 DP (capacity={CAPACITY}):")
print(f"  {'Name':<6}  {'Fraction':>10}  {'Value taken':>12}")
for name, frac, val in fk_taken:
    print(f"  {name:<6}  {frac:>10.3f}  {val:>12.2f}")
wt = sum(next(w for n,w,v in BATCHES if n==name)*frac for name,frac,_ in fk_taken)
print(f"  Weight used      : {wt:.2f} / {CAPACITY}")
print(f"  Fractional total : {fk_val:.2f}")
print(f"  0/1 DP total     : {dp01_val:.2f}")
print(f"  Fractional ≥ 0/1 : {'✅ YES (divisibility helps)' if fk_val >= dp01_val - 0.01 else '❌'}")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " PRODUCTION OPTIMISER — DP & GREEDY SUITE".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  MAX PRODUCT SUBARRAY (DP)" + " "*(W-27) + "║")
print(f"║  {'Array length':<28}: {N_PROD:<26}║")
print(f"║  {'Max product':<28}: {mp_val:<26}║")
print(f"║  {'Subarray':<28}: {str(mp_sub):<26}║")
print(f"║  {'Correct (vs brute force)':<28}: {'YES ✅' if mp_val == bf_max else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  COIN CHANGE — DP vs GREEDY" + " "*(W-28) + "║")
print(f"║  {'Denominations':<28}: {str(COINS):<26}║")
print(f"║  {'Target':<28}: {COIN_TGT:<26}║")
print(f"║  {'DP min coins':<28}: {dp_count:<26}║")
print(f"║  {'Greedy coins':<28}: {gr_count:<26}║")
print(f"║  {'DP optimal':<28}: {'YES ✅' if dp_count <= gr_count else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  ACTIVITY SELECTION (GREEDY)" + " "*(W-29) + "║")
print(f"║  {'Runs (n)':<28}: {N_RUNS:<26}║")
print(f"║  {'Runs selected':<28}: {len(act_sel):<26}║")
print(f"║  {'Non-overlapping':<28}: {'YES ✅' if sel_ok else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  FRACTIONAL KNAPSACK vs 0/1 DP" + " "*(W-31) + "║")
print(f"║  {'Batches (n)':<28}: {N_BATCH:<26}║")
print(f"║  {'Capacity':<28}: {CAPACITY:<26}║")
print(f"║  {'Fractional value':<28}: {fk_val:.2f}{'':<22}║")
print(f"║  {'0/1 DP value':<28}: {dp01_val:.2f}{'':<22}║")
print(f"║  {'Fractional ≥ 0/1':<28}: {'YES ✅' if fk_val >= dp01_val - 0.01 else 'NO ❌':<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What do they tell you about the algorithms in this project?

*Your answer here:*

---

**Q2 — Which technique was most surprising, and why?**
Pick one algorithm from this project. Explain why the approach works — and give one scenario where it would *fail* or need modification.

*Your answer here:*

---

**Q3 — How does this project connect to earlier weeks?**
Name at least two specific weeks whose topics appear in this project and explain the connection.

*Your answer here:*
